# Green NLP Benchmark Lab — Notebook 3: Retrieval — coding workshop (beginner)

**Content warning:** Part 3 uses sentences from *Speechwriter* (Paulo Naseiro), including
two with the novel's own coarse language (English "fuck"/"shit"-level swearing). They are
kept unfiltered because idiomatic profanity is itself a translation challenge.

**Session 3 of 3.** Sessions 1 and 2 built a QA pipeline, measured its baseline cost, and
then changed one thing: precision. Today changes one more thing. Instead of handing the
model every document for every question, a retriever first picks out the relevant ones.

The corpus has also grown from 8 documents to 26: the original 8, plus most of the papers
and facts cited across all three sessions. With 8 documents, stuffing everything into the
prompt was already cheap. With 26, the gap between "give the model everything" and "give
the model only what's relevant" becomes large enough to measure, and this notebook measures
both on the same corpus.

**Before you start:**

1. `Runtime` → `Change runtime type` → **T4 GPU** → `Save`.
2. Open the Files sidebar (folder icon, left) and upload the `leaderboard.csv`,
   `mt_leaderboard.csv` and `mt_eval_leaderboard.csv` you saved in Sessions 1 and 2. This
   notebook appends to them. Without them, it starts fresh files.

Expected time: 60–75 minutes for Part 1 (retrieval-augmented QA), plus 50–70 minutes for
Parts 2 and 3 (sparse vs. dense retrieval; retrieval-augmented translation and its
evaluation) if your session has time for them.

## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or put it straight into your exercise cell: the first line of the cell is `# %%solution N`.
   Delete the `# ` at its start and run the cell; the solution replaces the code in the cell.
   (This works once you have run the cell right below these instructions.)

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

In [ ]:
# @title Run this cell once: it lets you put a solution into its exercise cell (point 4 above).
from IPython.core.magic import register_cell_magic

SOLUTIONS = {
    1: "TOP_K = 2\n\ndef retrieve(question, k=TOP_K):\n    q_emb = embedder.encode([question], normalize_embeddings=True)[0]\n    sims = doc_embeddings @ q_emb\n    top_idx = np.argsort(-sims)[:k]\n    return [(DOC_NAMES[i], float(sims[i])) for i in top_idx]\n\n\nprint(\"Q:\", QA_PAIRS[0][\"question\"])\nfor name, score in retrieve(QA_PAIRS[0][\"question\"]):\n    print(f\"  {score:.3f}  {name}\")\nprint(\"Correct source:\", QA_PAIRS[0][\"source\"])",
    2: "def build_prompt(question, doc_names):\n    if not doc_names:\n        messages = [\n            {\"role\": \"system\",\n             \"content\": \"You are a concise assistant. Answer the question in one short sentence.\"},\n            {\"role\": \"user\", \"content\": f\"Question: {question}\"},\n        ]\n        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    context = \"\\n\\n\".join(f\"[{name}]\\n{CORPUS[name].strip()}\" for name in doc_names)\n    messages = [\n        {\n            \"role\": \"system\",\n            \"content\": (\n                \"You are a concise assistant. Answer the question using ONLY the \"\n                \"documents provided below. Answer in one short sentence.\"\n            ),\n        },\n        {\n            \"role\": \"user\",\n            \"content\": f\"Documents:\\n{context}\\n\\nQuestion: {question}\",\n        },\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef answer_question(question, doc_names, max_new_tokens=60):\n    prompt = build_prompt(question, doc_names)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    n_prompt_tokens = inputs[\"input_ids\"].shape[1]\n    answer = tokenizer.decode(output[0][n_prompt_tokens:], skip_special_tokens=True).strip()\n    return answer, n_prompt_tokens\n\n\nq = QA_PAIRS[0][\"question\"]\nretrieved_names = [name for name, _ in retrieve(q)]\nanswer, n_tokens = answer_question(q, retrieved_names)\n_, n_tokens_all = answer_question(q, DOC_NAMES, max_new_tokens=1)\nprint(\"Q:\", q)\nprint(\"A:\", answer)\nprint(f\"Prompt length: {n_tokens} tokens with {TOP_K} retrieved documents, \"\n      f\"{n_tokens_all} tokens with all {len(DOC_NAMES)}\")",
    3: "def run_quiz(project_name, context):\n    tracker = make_tracker(project_name)\n    rows = []\n    tracker.start()\n    start_time = time.time()\n    for pair in QA_PAIRS:\n        if context == \"retrieval\":\n            doc_names = [name for name, _ in retrieve(pair[\"question\"])]\n        elif context == \"full\":\n            doc_names = DOC_NAMES\n        else:\n            doc_names = []\n        answer, n_prompt_tokens = answer_question(pair[\"question\"], doc_names)\n        rows.append({\n            **pair,\n            \"retrieved\": doc_names if context == \"retrieval\" else None,\n            \"retrieval_correct\": pair[\"source\"] in doc_names,\n            \"answer\": answer,\n            \"prompt_tokens\": n_prompt_tokens,\n        })\n    elapsed = time.time() - start_time\n    emissions_kg = tracker.stop()\n    print(f\"{project_name}: {len(QA_PAIRS)} questions in {elapsed:.1f} s, \"\n          f\"{emissions_kg * 1000:.4f} g CO2eq\")\n    return {\"rows\": rows, \"elapsed\": elapsed, \"emissions_kg\": emissions_kg,\n            \"energy_kwh\": energy_kwh_of(tracker)}\n\n\nrun_closed = run_quiz(\"green-nlp-closed-book\", context=\"closed-book\")\nrun_full = run_quiz(\"green-nlp-full-context-26\", context=\"full\")\nrun_retrieval = run_quiz(\"green-nlp-retrieval\", context=\"retrieval\")",
    4: "def score_answer(answer, reference_list):\n    answer_lower = answer.lower()\n    return any(ref.lower() in answer_lower for ref in reference_list)\n\n\nfor run in (run_closed, run_full, run_retrieval):\n    for r in run[\"rows\"]:\n        r[\"correct\"] = score_answer(r[\"answer\"], r[\"reference\"])\n    rows = run[\"rows\"]\n    run[\"accuracy\"] = np.mean([r[\"correct\"] for r in rows])\n    run[\"accuracy_session1_questions\"] = np.mean([r[\"correct\"] for r in rows[:N_SESSION1_QUESTIONS]])\n    run[\"mean_prompt_tokens\"] = np.mean([r[\"prompt_tokens\"] for r in rows])\n\nrun_retrieval[\"retrieval_accuracy\"] = np.mean([r[\"retrieval_correct\"] for r in run_retrieval[\"rows\"]])\n\nfor r_closed, r_full, r_ret in zip(run_closed[\"rows\"], run_full[\"rows\"], run_retrieval[\"rows\"]):\n    print(f\"Q: {r_ret['question']}\")\n    print(f\"   closed book  {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\")\n    print(f\"   full context {'✓' if r_full['correct'] else '✗'}  {r_full['answer']}\")\n    print(f\"   retrieval    {'✓' if r_ret['correct'] else '✗'}  {r_ret['answer']}\")\n    if not r_ret[\"retrieval_correct\"]:\n        print(f\"   retriever missed: fetched {r_ret['retrieved']}, answer is in {r_ret['source']}\")\n    print()\n\nsummary = pd.DataFrame({\n    \"closed book (no docs)\": {\n        \"answer accuracy\": run_closed[\"accuracy\"],\n        \"accuracy, Notebook 1's 8 questions\": run_closed[\"accuracy_session1_questions\"],\n        \"retrieval accuracy\": None,\n        \"mean prompt tokens\": run_closed[\"mean_prompt_tokens\"],\n        \"g CO2eq per question\": run_closed[\"emissions_kg\"] * 1000 / len(QA_PAIRS),\n    },\n    \"full context (26 docs)\": {\n        \"answer accuracy\": run_full[\"accuracy\"],\n        \"accuracy, Notebook 1's 8 questions\": run_full[\"accuracy_session1_questions\"],\n        \"retrieval accuracy\": None,\n        \"mean prompt tokens\": run_full[\"mean_prompt_tokens\"],\n        \"g CO2eq per question\": run_full[\"emissions_kg\"] * 1000 / len(QA_PAIRS),\n    },\n    f\"retrieval (top {TOP_K})\": {\n        \"answer accuracy\": run_retrieval[\"accuracy\"],\n        \"accuracy, Notebook 1's 8 questions\": run_retrieval[\"accuracy_session1_questions\"],\n        \"retrieval accuracy\": run_retrieval[\"retrieval_accuracy\"],\n        \"mean prompt tokens\": run_retrieval[\"mean_prompt_tokens\"],\n        \"g CO2eq per question\": run_retrieval[\"emissions_kg\"] * 1000 / len(QA_PAIRS),\n    },\n})\nsummary",
    5: "tfidf = TfidfVectorizer(stop_words=\"english\")\ndoc_tfidf = tfidf.fit_transform(DOC_TEXTS)\n\ndef retrieve_sparse(question, k=TOP_K):\n    q_tfidf = tfidf.transform([question])\n    sims = cosine_similarity(q_tfidf, doc_tfidf)[0]\n    top_idx = np.argsort(-sims)[:k]\n    return [(DOC_NAMES[i], float(sims[i])) for i in top_idx]\n\n\ndef time_per_query_ms(retriever):\n    start = time.perf_counter()\n    for pair in QA_PAIRS:\n        retriever(pair[\"question\"])\n    return (time.perf_counter() - start) * 1000 / len(QA_PAIRS)\n\n\ncomparison = []\nfor pair in QA_PAIRS:\n    sparse_hits = [name for name, _ in retrieve_sparse(pair[\"question\"])]\n    dense_hits = [name for name, _ in retrieve(pair[\"question\"])]\n    comparison.append({\n        \"question\": pair[\"question\"],\n        \"source\": pair[\"source\"],\n        \"sparse\": sparse_hits,\n        \"sparse_ok\": pair[\"source\"] in sparse_hits,\n        \"dense\": dense_hits,\n        \"dense_ok\": pair[\"source\"] in dense_hits,\n    })\ncomparison = pd.DataFrame(comparison)\n\nprint(f\"Sparse (TF-IDF) retrieval accuracy:   {comparison['sparse_ok'].mean():.0%}\")\nprint(f\"Dense (embedding) retrieval accuracy: {comparison['dense_ok'].mean():.0%}\")\nprint(f\"Sparse time per query: {time_per_query_ms(retrieve_sparse):.2f} ms\")\nprint(f\"Dense time per query:  {time_per_query_ms(retrieve):.2f} ms\")",
    6: "TRANSLATION_MEMORY = [\n    {\"gl\": \"O meu curmán Fernando aguantaba o trebón que lle estaba a caer por riba.\",\n     \"en\": \"My cousin Fernando was weathering the storm breaking over his head.\"},\n    {\"gl\": \"Estou rodeado de merda, carallo! Veño dar un discurso do carallo, e o carallo do discurso está na mesa dalgún pringhao papando hostias!\",\n     \"en\": \"I'm surrounded by morons, damn it! I come here to give a fucking speech, and it is sitting on some loser's desk twiddling his fucking thumbs!\"},\n    {\"gl\": \"Un paso máis atrás, o alcalde suaba puro medo.\",\n     \"en\": \"One step behind him, the mayor was sweating pure fear.\"},\n    {\"gl\": \"Pero que dúas horas nin que merda! Ti toleas! Como vou ter dúas horas aí a esa xente papando hostias!\",\n     \"en\": \"What do you mean, two fucking hours? Are you fucking crazy? How am I supposed to keep all those people there twiddling their fucking thumbs for two hours?\"},\n    {\"gl\": \"Xuro que fun capaz de ler o pensamento do alcalde naquel momento.\",\n     \"en\": \"I swear I could read the mayor's thoughts at that moment.\"},\n    {\"gl\": \"Á noitiña, xa máis calmo, pagou os cubatas.\",\n     \"en\": \"That evening, once he had calmed down, Fernando bought me some rum and cokes.\"},\n]\n\nFULL_TEXT_GL = (\n    \"Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e \"\n    \"refráns, dicía algo así como \\\"un é escravo do que di e dono do que cala\\\". Sempre me \"\n    \"pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, \"\n    \"aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida.\"\n)\nREFERENCE_EN = (\n    \"Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings \"\n    \"and proverbs; she used to say something like, 'You're a slave to what you say, and \"\n    \"master of what you keep to yourself.' It always struck me as a wise saying, and maybe \"\n    \"I should have listened to it on a few occasions — the rare ones that, even so, ended up \"\n    \"sharply changing the course of my life.\"\n)\n\ntm_embeddings = embedder.encode([pair[\"gl\"] for pair in TRANSLATION_MEMORY],\n                                normalize_embeddings=True)\n\ndef retrieve_translation_example(source_text):\n    q_emb = embedder.encode([source_text], normalize_embeddings=True)[0]\n    sims = tm_embeddings @ q_emb\n    best_idx = int(np.argmax(sims))\n    return TRANSLATION_MEMORY[best_idx], sims\n\n\nbest_example, tm_sims = retrieve_translation_example(FULL_TEXT_GL)\nfor pair, sim in sorted(zip(TRANSLATION_MEMORY, tm_sims), key=lambda x: -x[1]):\n    print(f\"{sim:.3f}  {pair['gl']}\")\nprint(f\"\\nRetrieved: {best_example['gl']}\\n       -> {best_example['en']}\")",
    7: "def append_rows(path, rows):\n    if os.path.exists(path):\n        table = pd.read_csv(path)\n        print(f\"Appending to {path} ({len(table)} earlier row(s)).\")\n        table = pd.concat([table, pd.DataFrame(rows)], ignore_index=True)\n    else:\n        print(f\"No {path} found. Starting a fresh one.\")\n        table = pd.DataFrame(rows)\n    table.to_csv(path, index=False)\n    return table\n\n\naugmented = mt_runs[\"retrieval-augmented\"]\nAUGMENTED = \"general-llm-retrieval-augmented\"\n\nmt_leaderboard = append_rows(\"mt_leaderboard.csv\", [{\n    \"task\": \"translation\",\n    \"era\": AUGMENTED,\n    \"model\": MODEL_NAME,\n    \"duration_sec\": augmented[\"elapsed\"],\n    \"energy_kwh\": augmented[\"energy_kwh\"],\n    \"emissions_kg_co2eq\": augmented[\"emissions_kg\"],\n    \"bleu_score\": eval_runs[\"bleu\"][\"scores\"][AUGMENTED],\n    \"config\": \"retrieval-augmented-fp16\",\n    \"translation\": augmented[\"translation\"],\n}])\n\nmt_eval_leaderboard = append_rows(\"mt_eval_leaderboard.csv\", [\n    {\"metric\": metric, \"candidate\": AUGMENTED, \"score\": run[\"scores\"][AUGMENTED],\n     \"duration_sec\": run[\"elapsed\"], \"energy_kwh\": run[\"energy_kwh\"],\n     \"emissions_kg_co2eq\": run[\"emissions_kg\"]}\n    for metric, run in eval_runs.items() if AUGMENTED in run[\"scores\"]\n])",
}


@register_cell_magic
def solution(line, cell):
    """`%%solution N` as the first line of a cell: the solution to Exercise N replaces the cell's code."""
    n = int(line.split()[0])
    get_ipython().set_next_input(SOLUTIONS[n], replace=True)
    print(f"The solution to Exercise {n} is now in this cell. Run the cell, then its check cell.")

## Step 0 — Install and import

Colab already ships `transformers`, `accelerate`, `sentence-transformers`, `scikit-learn`,
`pandas` and `matplotlib`. Three libraries are missing: `codecarbon` (energy and CO2
measurement, as in Sessions 1 and 2), and `sacrebleu` and `bert-score` (translation
metrics in Part 3).

In [ ]:
!pip install -q codecarbon sacrebleu bert-score

In [ ]:
import os
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sacrebleu
import torch
from codecarbon import EmissionsTracker
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook still runs, only slower. "
          "Check Runtime > Change runtime type > T4 GPU.")

In [ ]:

def make_tracker(project_name):
    return EmissionsTracker(
        project_name=project_name,
        measure_power_secs=1,
        save_to_file=False,
        log_level="warning",
    )


def energy_kwh_of(tracker):
    try:
        return float(tracker.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None

## Step 1 — The corpus, grown to 26 documents

The first 8 are Notebook 1's corpus, unchanged. The other 18 are short original summaries
of papers and facts this course has cited: the "5 cars" correction, the attention paper,
the RAG paper itself, and so on. Same format as before: one short paragraph, usually ending
with a `Source:` line.

In [ ]:
CORPUS = {
    "training_vs_inference": """
Training a large language model is a one-time, very expensive event: the organization
that builds the model spends a large amount of compute once. Inference — actually using
the trained model to answer a query — is comparatively cheap per request, but it happens
every single time anyone uses the model. Once a model is deployed and used by many people,
the accumulated cost of inference tends to dominate the model's total lifetime footprint,
even though any single inference looks negligible next to the training run.
Source: Patterson et al., "Carbon Emissions and Large Neural Network Training," 2021
(arXiv:2104.10350). Training GPT-3 was estimated to emit 552 metric tons of CO2-equivalent,
using about 1,287 megawatt-hours of electricity.
""",
    "data_centers": """
Data centers are the physical facilities that house the servers running most of the
world's AI models. According to the International Energy Agency, global data center
electricity consumption was about 415 terawatt-hours in 2024, roughly 1.5 percent of all
electricity consumed worldwide that year. That demand is growing fast: electricity use by
data centers rose by about 17 percent in 2025 alone, driven mostly by AI workloads, and is
projected to approach 950 terawatt-hours by 2030, or about 3 percent of global electricity
demand.
Source: International Energy Agency, "Energy and AI" and related data centre electricity
reports, 2024-2025.
""",
    "quantization_basics": """
Neural network weights are normally stored as 32-bit floating point numbers (FP32).
Quantization reduces the number of bits used to represent each weight — for example to
16-bit (FP16/BF16), 8-bit integers (INT8), or even lower. Fewer bits per number means a
smaller model file and less memory traffic when running it, at the cost of some numerical
precision. A useful analogy: quantizing a model is like reducing the number of colors used
to represent an image — the picture is still recognizable, but some fine detail is lost.
Whether this actually saves energy in practice depends on the hardware: it helps most when
the bottleneck is moving data in and out of memory, not raw computation.
Source: Grootendorst, "A Visual Guide to Quantization," 2024.
""",
    "measuring_carbon": """
Estimating the carbon footprint of running a model requires several assumptions: how much
electricity the hardware draws, the Power Usage Effectiveness (PUE) of the data center
(extra power spent on cooling and overhead), and the carbon intensity of the local
electricity grid — how much CO2 is emitted per unit of electricity, which varies a lot by
region and by time of day. Because of all these assumptions, published carbon estimates can
vary widely, and some influential early estimates have later been revised — sometimes by an
order of magnitude — once other researchers re-examined the methodology. This is one reason
this course emphasizes measuring your own numbers.
""",
    "inference_cost_task_type": """
Models doing the same task can differ widely in what they cost to run. A 2024 study
compared task-specific models (built and trained for one narrow job) against multi-purpose
generative models (like general-purpose chatbots) performing the exact same tasks. For
extractive question answering, the most efficient task-specific models emitted around 0.3
grams of CO2-equivalent per 1,000 inferences, while general-purpose generative models
emitted around 10 grams of CO2-equivalent per 1,000 inferences for the same task — roughly
33 times more. The gap was consistent across several other tasks too, even when
controlling for the number of parameters.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of AI
Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
    "rag_concept": """
Retrieval-Augmented Generation, or RAG, is a technique where a model is given relevant
external text — retrieved from a document collection — as part of its input at query time,
instead of relying only on what it "learned" during training. This lets a smaller, cheaper
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model on that information. The retrieval step
usually uses a separate, lightweight embedding model to find which documents are relevant
before the language model ever sees them.
""",
    "small_models_trend": """
Between 2024 and 2026, there has been a strong trend toward small, efficient open-weight
language models (often called SLMs, small language models) in the 1-4 billion parameter
range, designed to run on a single consumer GPU or even a laptop CPU after quantization.
These models are trained to be very capable for their size, and combined with quantization
and retrieval, they let many applications avoid depending on a large, general-purpose
model at all — often at a fraction of the inference cost, if the task doesn't need the
breadth of a much bigger model.
""",
    "transformer_basics": """
Modern language models are built from transformer layers. Inside each layer,
"self-attention" lets every position in the input sequence gather information from every
other position, which is what allows the model to relate words like "it" back to the noun
it refers to several words earlier. This mechanism is powerful but has a computational cost
that grows faster than the length of the input (roughly quadratically with sequence
length), and more parameters mean more numbers to multiply and move through memory for
every single token generated. This is the underlying reason why both model size and input
length are direct levers on energy cost.
""",
    "nas_architecture_search": """
A widely repeated 2019 claim held that training a particular large NLP model produced
roughly as much lifetime CO2-equivalent as five cars. The model came from a neural
architecture search: instead of a human hand-designing a network's structure, an
algorithm trains and evaluates thousands of candidate architectures before selecting a
winner, multiplying the effective training cost many times over. The search itself was
hunting for a better machine translation architecture. The original "five cars" estimate
was later shown to be roughly 10 times too high, once other researchers re-examined the
hardware and power assumptions behind it.
Source: Strubell et al., "Energy and Policy Considerations for Deep Learning in NLP," ACL
2019; So, Le & Liang, "The Evolved Transformer," 2019.
""",
    "nlp_market_size": """
Market-research estimates of the global NLP industry's size vary widely even for the same
year. One widely-cited 2025 estimate puts the global NLP market at roughly $37 billion,
projected to reach about $193 billion by 2034 — about 20 percent annual growth. Other
firms estimate the same year's market anywhere from about $25 billion to well above $60
billion. The spread exists because "NLP market" is a loosely defined category: analysts
include or exclude voice assistants, chatbot platforms, search, and broader "AI" spending
differently, and each firm's method is rarely published in full.
Source: Fortune Business Insights, "Natural Language Processing Market," 2025.
""",
    "mt_history": """
Machine translation is one of AI's oldest stated goals. The first public demonstration was
the Georgetown-IBM experiment in 1954, translating a small set of Russian sentences into
English, received with considerable, later unrealistic, optimism about how soon
translation would be "solved." In 1966 the U.S. government's ALPAC report concluded
machine translation was falling short of its promises and recommended cutting funding,
setting back American MT research roughly a decade. Progress resumed with rule-based
systems, then statistical systems that learned patterns directly from translated text;
Google Translate launched on statistical MT in 2006.
Source: Hutchins, "Machine Translation: A Concise History," 2007.
""",
    "attention_origin": """
Early neural machine translation systems compressed an entire source sentence into a
single fixed-size vector before generating a translation from that one vector alone — a
bottleneck that hurt quality as sentences got longer, since detail from earlier in the
sentence was easily lost. Bahdanau, Cho and Bengio fixed this in 2015 by letting the
decoder look back at every word of the source sentence at each generation step and learn
which source words matter most for the word currently being produced. They called this
mechanism attention. It was invented specifically to solve a machine translation problem,
years before it became the core building block of modern language models.
Source: Bahdanau, Cho & Bengio, "Neural Machine Translation by Jointly Learning to Align
and Translate," ICLR 2015.
""",
    "transformer_self_attention": """
In 2017, Vaswani and colleagues showed that the attention mechanism originally built for
translating between two languages also works within a single sentence: every word can
attend to every other word in the same sentence, including itself, a pattern called
self-attention. This lets a model resolve relationships like a pronoun referring back to a
noun several words earlier, without any hand-written grammatical rule. Their architecture,
the transformer, built entirely around this mechanism plus a small feed-forward network
per layer, is the architecture behind essentially every modern large language model.
Source: Vaswani et al., "Attention Is All You Need," NeurIPS 2017.
""",
    "skipgram_embeddings": """
Word embeddings — dense numerical vectors that place semantically similar words close
together — are learned from text. The skip-gram model, introduced by Mikolov and
colleagues, trains a network on a simple task: given one word, predict the words that tend
to appear near it in real text. The network's internal weight matrix, learned purely to
get better at that prediction task, becomes the word embeddings as a side effect. Nobody
tells the model what "similar" means; words that occur in similar contexts end up needing
similar internal representations to succeed at the same prediction task.
Source: Mikolov et al., "Efficient Estimation of Word Representations in Vector Space,"
2013.
""",
    "gpt_scaling": """
Until the late 2010s, NLP typically meant a separate, purpose-built model for each task:
one system for translation, another for summarization, another for sentiment. Radford and
colleagues showed in 2019, with GPT-2, that a single model trained only to predict the
next word of text could be redirected to many different tasks just by phrasing the request
differently — "prompting" it — without retraining it. Brown and colleagues confirmed
this worked at much larger scale in 2020 with GPT-3. This is broadly considered the moment
public conversation stopped clearly distinguishing "NLP" from "AI."
Source: Radford et al., "Language Models are Unsupervised Multitask Learners," 2019; Brown
et al., "Language Models are Few-Shot Learners," NeurIPS 2020.
""",
    "galician_llm_gap": """
Galician's presence in the training data of mainstream large language models has been
described in the research literature as "almost residual" — a very small fraction,
sitting close enough to Spanish and Portuguese that general-purpose models tend to blur
the three together instead of reliably producing correct, distinctly Galician text. One
response has been to build smaller, dedicated models: a 1.3-billion-parameter model
trained specifically on Galician and Portuguese text outperformed a much larger
general-purpose base model specifically at producing correct Galician.
Source: Gamallo et al., "A Galician-Portuguese Generative Model," EPIA 2024; Proxecto Nós,
Universidade de Santiago de Compostela.
""",
    "green_mt_knight": """
Some Green NLP research targets machine translation directly. One recent MT-specific
efficiency recipe, nicknamed "The Green KNIGHT," reached up to 140 times faster inference
and about 98.7 percent less energy than a baseline system, for roughly a 10 to 12 percent
relative cost in translation quality. Efficiency techniques are frequently developed and
tested in machine translation before being generalized to language models more broadly.
Source: Guta, Petrick & Polák, "The Green KNIGHT," EMNLP Findings 2025.
""",
    "mt_quantization": """
Quantization — reducing the number of bits used to store and compute with a model's
numbers — has been studied specifically in the context of machine translation systems.
Testing an efficiency technique on the specific task it will serve is consistent with this
course's broader theme: measure the technique on the task you actually care about.
Source: Shterionov & Vanmassenhove, 2022.
""",
    "quantization_energy_quality": """
A 2025 study systematically measured quantization's effects across several large language
models on production-grade GPUs. Quantization generally saved energy — up to about 30
percent versus full precision — and in some tests cut latency by as much as 70 percent.
But pushed too far, quality did not degrade gently: on a code-generation benchmark, one
13-billion-parameter model lost up to 92 percent of its pass rate under aggressive
quantization. The same study found that smaller models are more vulnerable to this kind of
quality collapse than larger ones.
Source: Shi & Ding, "Systematic Characterization of LLM Quantization: A Performance,
Energy, and Quality Perspective," 2025 (arXiv:2508.16712).
""",
    "quantization_hardware_dependency": """
A separate 2025 study tested 28 quantized language models, including the Qwen2.5 family,
on edge hardware. Its finding: 8-bit and 4-bit quantization roughly halved energy use
with only a modest accuracy cost, but only when the underlying hardware actually had
efficient support for low-precision arithmetic. Pushing further, to 3-bit quantization,
sometimes increased energy use instead of decreasing it, because the hardware spent more
effort unpacking the unusual bit-width than it saved. The benefit of quantization depends
on the chip it runs on.
Source: Husom et al., "Sustainable LLM Inference for Edge AI," 2025 (arXiv:2504.03360).
""",
    "gpu_precision_throughput": """
On many GPUs, lower numerical precision also runs faster, on top of shrinking file size
and memory use, because the hardware has separate circuits tuned for different
bit-widths. The NVIDIA T4 GPU, commonly available on free-tier cloud notebooks, has a peak
throughput of about 8.1 TFLOPS at 32-bit floating point, 65 TFLOPS at 16-bit, 130 TOPS at
8-bit integer, and 260 TOPS at 4-bit integer — roughly doubling with each step down in
precision, on the exact same physical chip.
Source: NVIDIA T4 Tensor Core GPU datasheet.
""",
    "rag_paper": """
The technique now generally called retrieval-augmented generation was formalized in a 2020
paper: instead of relying only on facts baked into a model's parameters during training,
retrieve relevant external text from a document collection at query time and condition the
model's generated answer on that retrieved text. This lets a comparatively small, cheap
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model to memorize that information.
Source: Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks,"
NeurIPS 2020 (arXiv:2005.11401).
""",
    "rag_vs_long_context": """
A 2024 study directly compared retrieval-augmented generation against long-context
prompting — feeding the whole text into a model with a large enough context window —
using three strong language models: GPT-4o, GPT-3.5-Turbo and Gemini-1.5-Pro. For all
three, long-context prompting gave better answers on average. But retrieval read far less
text: about 1,500 tokens of retrieved passages per question, against 10,000 to 100,000
tokens for the full texts. For 63 percent of the queries, the two approaches produced
identical answers. A combined method, Self-Route, tries retrieval first and switches to
long context only when the model judges the retrieved text insufficient; it cut cost by
65 percent for Gemini-1.5-Pro and 39 percent for GPT-4o while staying close to
long-context quality.
Source: Li, Li, Zhang, Mei & Bendersky, "Retrieval Augmented Generation or Long-Context
LLMs? A Comprehensive Study and Hybrid Approach," EMNLP 2024 Industry Track
(arXiv:2407.16833).
""",
    "knn_mt": """
Retrieval has also been applied directly to machine translation. One method augments an
already-trained translation model with a large datastore of previously translated example
sentence pairs. At each step of generating a translation, the system looks up the nearest
neighbors in that datastore and blends their suggested next word in with the trained
model's own prediction. This lets a translation system incorporate new examples without
retraining, in the same spirit as retrieval-augmented generation for question answering,
applied to translation instead.
Source: Khandelwal, Fan, Jurafsky, Zettlemoyer & Lewis, "Nearest Neighbor Machine
Translation," ICLR 2021.
""",
    "water_footprint": """
Electricity is only one of the resources AI systems consume. Data centers also use water,
mainly for cooling. Google has estimated that a single prompt to its Gemini model consumes
about 0.26 milliliters of water — roughly five drops. That figure is small per prompt but,
like electricity, scales with the number of queries a system serves, and water use is
generally reported and discussed far less often than energy or carbon figures are.
Source: Google, water-footprint estimate for Gemini prompts.
""",
    "image_generation_cost": """
Which task a model performs is itself one of the biggest levers on energy cost,
independent of any efficiency technique applied to the model. A 2024 study found that
image generation averaged about 2.9 kilowatt-hours per 1,000 inferences — roughly 1,450
times the energy of the cheapest text-based task measured in the same study. Choosing a
smaller, cheaper task, or a task-specific tool instead of a general-purpose one, can matter
as much as any single optimization applied afterward.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of
AI Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
}

for name, text in CORPUS.items():
    print(f"{name:34s} {len(text.split()):4d} words")
print(f"\nTotal documents: {len(CORPUS)}, "
      f"total words: {sum(len(t.split()) for t in CORPUS.values())}")

## Step 2 — The quiz

26 questions: Notebook 1's eight, word for word, followed by one question for each new
document. Same substring-match scoring as Notebook 1.

Each question now also records the document its answer lives in (`"source"`). That makes a
new measurement possible: did the retriever find the *right document*? Notebook 1's
baseline saw every document, so it could never miss one. A retriever can.

In [ ]:
QA_PAIRS = [
    {
        "question": "According to Patterson et al. (2021), how many metric tons of CO2-equivalent did training GPT-3 emit?",
        "reference": ["552"],
        "source": "training_vs_inference",
    },
    {
        "question": "Per the IEA, roughly what percentage of global electricity consumption did data centers represent in 2024?",
        "reference": ["1.5"],
        "source": "data_centers",
    },
    {
        "question": "What does the acronym PUE stand for?",
        "reference": ["power usage effectiveness"],
        "source": "measuring_carbon",
    },
    {
        "question": "What does carbon footprint estimation depend on besides hardware power draw and grid carbon intensity?",
        "reference": ["pue", "power usage effectiveness"],
        "source": "measuring_carbon",
    },
    {
        "question": "In the 2024 study on inference costs, about how many grams of CO2-equivalent do the most efficient task-specific models emit per 1,000 extractive question-answering inferences?",
        "reference": ["0.3"],
        "source": "inference_cost_task_type",
    },
    {
        "question": "What is the name of the technique where a model is given retrieved external text at query time instead of relying only on what it learned during training?",
        "reference": ["retrieval-augmented generation", "rag"],
        "source": "rag_concept",
    },
    {
        "question": "What size range (in billions of parameters) do small language models (SLMs) typically fall into?",
        "reference": ["1-4", "1 to 4", "1-4 billion"],
        "source": "small_models_trend",
    },
    {
        "question": "What mechanism inside a transformer layer lets every position gather information from every other position?",
        "reference": ["self-attention", "attention"],
        "source": "transformer_basics",
    },
    {
        "question": "The original '5 cars' carbon estimate for the architecture-search model was later shown to be roughly how many times too high?",
        "reference": ["10", "ten"],
        "source": "nas_architecture_search",
    },
    {
        "question": "Per Fortune Business Insights, roughly what is the projected global NLP market size by 2034?",
        "reference": ["193"],
        "source": "nlp_market_size",
    },
    {
        "question": "In what year did the Georgetown-IBM machine translation demonstration take place?",
        "reference": ["1954"],
        "source": "mt_history",
    },
    {
        "question": "In what year did Bahdanau, Cho and Bengio introduce the attention mechanism to fix early neural MT's single-vector bottleneck?",
        "reference": ["2015"],
        "source": "attention_origin",
    },
    {
        "question": "What is the name of the architecture introduced by Vaswani et al. in 2017, built entirely around self-attention?",
        "reference": ["transformer"],
        "source": "transformer_self_attention",
    },
    {
        "question": "What is the name of the model introduced by Mikolov et al. that produces word embeddings as a byproduct of predicting nearby words?",
        "reference": ["skip-gram", "skipgram"],
        "source": "skipgram_embeddings",
    },
    {
        "question": "In what year did Brown et al. show GPT-3 could perform many tasks via prompting at large scale?",
        "reference": ["2020"],
        "source": "gpt_scaling",
    },
    {
        "question": "How many billion parameters does the dedicated Galician-Portuguese model described in Gamallo et al. have?",
        "reference": ["1.3"],
        "source": "galician_llm_gap",
    },
    {
        "question": "According to the Green KNIGHT study, roughly what percentage energy reduction did their MT-specific recipe achieve?",
        "reference": ["98.7"],
        "source": "green_mt_knight",
    },
    {
        "question": "What NLP task was quantization tested on specifically, per Shterionov and Vanmassenhove (2022)?",
        "reference": ["machine translation", "mt"],
        "source": "mt_quantization",
    },
    {
        "question": "Per Shi and Ding (2025), up to what percentage drop in code-generation pass rate did aggressive quantization cause for a 13B model?",
        "reference": ["92"],
        "source": "quantization_energy_quality",
    },
    {
        "question": "Per Husom et al. (2025), which specific model family was tested for quantization energy efficiency on edge hardware?",
        "reference": ["qwen2.5", "qwen"],
        "source": "quantization_hardware_dependency",
    },
    {
        "question": "According to the NVIDIA T4 datasheet, what is the GPU's peak INT8 throughput in TOPS?",
        "reference": ["130"],
        "source": "gpu_precision_throughput",
    },
    {
        "question": "In what year did Lewis et al. introduce the original Retrieval-Augmented Generation (RAG) framework?",
        "reference": ["2020"],
        "source": "rag_paper",
    },
    {
        "question": "In the Li et al. (2024) study, for what percentage of queries did RAG and long-context prompting produce identical answers?",
        "reference": ["63"],
        "source": "rag_vs_long_context",
    },
    {
        "question": "What is the common short name for the retrieval-augmented machine translation method introduced by Khandelwal et al. in 2021?",
        "reference": ["knn-mt", "knn"],
        "source": "knn_mt",
    },
    {
        "question": "Roughly how many milliliters of water does Google estimate a single Gemini prompt consumes?",
        "reference": ["0.26"],
        "source": "water_footprint",
    },
    {
        "question": "Per Luccioni et al. (2024), roughly how many kilowatt-hours per 1,000 inferences does image generation average?",
        "reference": ["2.9"],
        "source": "image_generation_cost",
    },
]

N_SESSION1_QUESTIONS = 8

assert all(pair["source"] in CORPUS for pair in QA_PAIRS)
print(f"Number of questions: {len(QA_PAIRS)}")

## Step 3 — Load the model (same as Notebook 1)

Same `Qwen2.5-1.5B-Instruct` at the same 16-bit precision as Session 1's `baseline-fp16`
row, so model size and precision stay fixed. The only thing that changes is what the model
gets to read.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
N_PARAMS_BILLION = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Loaded {MODEL_NAME} ({N_PARAMS_BILLION:.2f}B parameters)")

## Step 4 — Load an embedding model, and embed the corpus once

The retriever uses `paraphrase-multilingual-MiniLM-L12-v2`, the multilingual sentence
encoder from Session 1's Embeddings Playground. Part 3 needs a multilingual encoder for
its Galician sentences, so one model covers both jobs.

Each of the 26 documents is embedded once, up front. That is the one-time cost of building
a retriever, separate from the per-question cost of using it, so it gets its own
measurement.

One property of this encoder matters later: it reads at most `max_seq_length` tokens
(128) and silently drops the rest. The cell below counts how many documents are longer
than that.

In [ ]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

DOC_NAMES = list(CORPUS)
DOC_TEXTS = [CORPUS[name].strip() for name in DOC_NAMES]

tracker_index = make_tracker("green-nlp-retrieval-index")
tracker_index.start()
doc_embeddings = embedder.encode(DOC_TEXTS, normalize_embeddings=True)
emissions_index_kg = tracker_index.stop()

print(f"Embedded {len(DOC_NAMES)} documents into {doc_embeddings.shape[1]}-dimensional vectors")
print(f"One-time indexing cost: {emissions_index_kg * 1000:.6f} g CO2eq")

doc_lengths = [len(embedder.tokenizer(text)["input_ids"]) for text in DOC_TEXTS]
n_truncated = sum(n > embedder.max_seq_length for n in doc_lengths)
print(f"\nEncoder window: {embedder.max_seq_length} tokens. "
      f"Documents longer than that: {n_truncated} of {len(DOC_TEXTS)} "
      f"(longest: {max(doc_lengths)} tokens).")

## Step 5 — Retrieve: rank documents by cosine similarity to the question

This is the mechanism from Session 1's cosine-similarity screen and Embeddings Playground,
applied to whole documents instead of single words: embed the question, compare it against
every document's embedding, keep the `TOP_K` most similar.

`TOP_K = 2` gives the retriever some slack: if the right document comes second, the model
still sees it. That is still under a tenth of the corpus per question.

The sanity check below uses the quiz's first question, and the right document is *not*
among the top 2. Keep that in mind: Part 2 explains why.

### ✏️ Exercise 1 — Dense retrieval

Complete `retrieve`: compute the cosine similarity of the question to every
document, and keep the indices of the `k` most similar documents, best first. All
embeddings are normalized, so a dot product is a cosine similarity.

<details><summary><b>Hints</b> (click to open)</summary>

- `doc_embeddings` has one row per document; `doc_embeddings @ q_emb` gives one score per row.
- `np.argsort` sorts from smallest to largest. Sort `-sims` to put the largest first.

</details>

In [ ]:
# %%solution 1   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
TOP_K = 2

def retrieve(question, k=TOP_K):
    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    sims = doc_embeddings @ ___
    top_idx = np.argsort(___)[:k]
    return [(DOC_NAMES[i], float(sims[i])) for i in top_idx]


print("Q:", QA_PAIRS[0]["question"])
for name, score in retrieve(QA_PAIRS[0]["question"]):
    print(f"  {score:.3f}  {name}")
print("Correct source:", QA_PAIRS[0]["source"])

In [ ]:
# Check for Exercise 1. Run this cell as is.
_hits = retrieve(QA_PAIRS[0]["question"], k=3)
assert len(_hits) == 3 and _hits[0][1] >= _hits[1][1] >= _hits[2][1], "best first"
assert retrieve(DOC_TEXTS[5], k=1)[0][0] == DOC_NAMES[5], "a document's own text should retrieve that document"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
TOP_K = 2

def retrieve(question, k=TOP_K):
    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    sims = doc_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:k]
    return [(DOC_NAMES[i], float(sims[i])) for i in top_idx]


print("Q:", QA_PAIRS[0]["question"])
for name, score in retrieve(QA_PAIRS[0]["question"]):
    print(f"  {score:.3f}  {name}")
print("Correct source:", QA_PAIRS[0]["source"])
```

</details>

## Step 6 — The pipeline: retrieve, then read

`build_prompt` is Notebook 1's, with one change: it takes a list of document names, so the
same function can build a prompt from the 2 retrieved documents, from all 26, or from none.
With documents, the system prompt is Notebook 1's. With no documents, the model is simply
asked the question and has to answer from what it learned in training. Same "answer in one
short sentence" instruction, same greedy decoding.

### ✏️ Exercise 2 — A prompt from the retrieved documents only

In `build_prompt`, build `context` from the documents named in `doc_names` only:
each as `[name]`, a newline, and its text, with a blank line between them.

<details><summary><b>Hints</b> (click to open)</summary>

- `CORPUS[name]` is the text of document `name`; `.strip()` trims the blank lines around it.

</details>

In [ ]:
# %%solution 2   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
def build_prompt(question, doc_names):
    if not doc_names:
        messages = [
            {"role": "system",
             "content": "You are a concise assistant. Answer the question in one short sentence."},
            {"role": "user", "content": f"Question: {question}"},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    context = "\n\n".join(f"[{name}]\n{___}" for name in ___)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, doc_names, max_new_tokens=60):
    prompt = build_prompt(question, doc_names)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_prompt_tokens = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(output[0][n_prompt_tokens:], skip_special_tokens=True).strip()
    return answer, n_prompt_tokens


q = QA_PAIRS[0]["question"]
retrieved_names = [name for name, _ in retrieve(q)]
answer, n_tokens = answer_question(q, retrieved_names)
_, n_tokens_all = answer_question(q, DOC_NAMES, max_new_tokens=1)
print("Q:", q)
print("A:", answer)
print(f"Prompt length: {n_tokens} tokens with {TOP_K} retrieved documents, "
      f"{n_tokens_all} tokens with all {len(DOC_NAMES)}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
_p = build_prompt("Q?", ["rag_concept"])
assert "[rag_concept]" in _p and CORPUS["rag_concept"].strip() in _p
assert "[data_centers]" not in _p, "only the documents in doc_names belong in the prompt"
assert n_tokens < n_tokens_all
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
def build_prompt(question, doc_names):
    if not doc_names:
        messages = [
            {"role": "system",
             "content": "You are a concise assistant. Answer the question in one short sentence."},
            {"role": "user", "content": f"Question: {question}"},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    context = "\n\n".join(f"[{name}]\n{CORPUS[name].strip()}" for name in doc_names)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, doc_names, max_new_tokens=60):
    prompt = build_prompt(question, doc_names)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_prompt_tokens = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(output[0][n_prompt_tokens:], skip_special_tokens=True).strip()
    return answer, n_prompt_tokens


q = QA_PAIRS[0]["question"]
retrieved_names = [name for name, _ in retrieve(q)]
answer, n_tokens = answer_question(q, retrieved_names)
_, n_tokens_all = answer_question(q, DOC_NAMES, max_new_tokens=1)
print("Q:", q)
print("A:", answer)
print(f"Prompt length: {n_tokens} tokens with {TOP_K} retrieved documents, "
      f"{n_tokens_all} tokens with all {len(DOC_NAMES)}")
```

</details>

## Step 7 — Measure: three runs over the same 26 questions

All three runs use the same model and the same questions. The only difference is what goes
into the prompt:

- **Closed book:** the question alone, no documents. What the model can answer from its
  training data. Notebook 1 measured this on its eight questions; Notebook 2 always handed
  the model the documents, which contain the answers.
- **Full context:** all 26 documents, every question. Notebook 1's pipeline on the bigger
  corpus.
- **Retrieval:** the `TOP_K` retrieved documents. Retrieval itself (embedding the question,
  ranking 26 documents) runs inside the tracker, so its cost is included.

For the retrieval run we also record whether the right document was retrieved
(`retrieval_correct`). A wrong final answer can now come from two places: the retriever
fetched the wrong document, or the model answered badly with the right one in front of it.

### ✏️ Exercise 3 — Three ways to fill the context

In `run_quiz`, choose `doc_names` for each question:

- `"retrieval"`: the names returned by `retrieve`;
- `"full"`: every document;
- anything else (closed book): no documents.

The cell then runs all three quizzes. If a check fails, fix the code and rerun this cell.

<details><summary><b>Hints</b> (click to open)</summary>

- `retrieve(question)` returns `(name, score)` pairs; keep the names.
- `DOC_NAMES` lists every document.

</details>

In [ ]:
# %%solution 3   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
def run_quiz(project_name, context):
    tracker = make_tracker(project_name)
    rows = []
    tracker.start()
    start_time = time.time()
    for pair in QA_PAIRS:
        if context == "retrieval":
            doc_names = [name for name, _ in ___]
        elif context == "full":
            doc_names = ___
        else:
            doc_names = ___
        answer, n_prompt_tokens = answer_question(pair["question"], doc_names)
        rows.append({
            **pair,
            "retrieved": doc_names if context == "retrieval" else None,
            "retrieval_correct": pair["source"] in doc_names,
            "answer": answer,
            "prompt_tokens": n_prompt_tokens,
        })
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    print(f"{project_name}: {len(QA_PAIRS)} questions in {elapsed:.1f} s, "
          f"{emissions_kg * 1000:.4f} g CO2eq")
    return {"rows": rows, "elapsed": elapsed, "emissions_kg": emissions_kg,
            "energy_kwh": energy_kwh_of(tracker)}


run_closed = run_quiz("green-nlp-closed-book", context="closed-book")
run_full = run_quiz("green-nlp-full-context-26", context="full")
run_retrieval = run_quiz("green-nlp-retrieval", context="retrieval")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert all(len(r["retrieved"]) == TOP_K for r in run_retrieval["rows"])
assert run_closed["rows"][0]["prompt_tokens"] < run_retrieval["rows"][0]["prompt_tokens"] \
    < run_full["rows"][0]["prompt_tokens"], "closed book < retrieval < full context, in prompt length"
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def run_quiz(project_name, context):
    tracker = make_tracker(project_name)
    rows = []
    tracker.start()
    start_time = time.time()
    for pair in QA_PAIRS:
        if context == "retrieval":
            doc_names = [name for name, _ in retrieve(pair["question"])]
        elif context == "full":
            doc_names = DOC_NAMES
        else:
            doc_names = []
        answer, n_prompt_tokens = answer_question(pair["question"], doc_names)
        rows.append({
            **pair,
            "retrieved": doc_names if context == "retrieval" else None,
            "retrieval_correct": pair["source"] in doc_names,
            "answer": answer,
            "prompt_tokens": n_prompt_tokens,
        })
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    print(f"{project_name}: {len(QA_PAIRS)} questions in {elapsed:.1f} s, "
          f"{emissions_kg * 1000:.4f} g CO2eq")
    return {"rows": rows, "elapsed": elapsed, "emissions_kg": emissions_kg,
            "energy_kwh": energy_kwh_of(tracker)}


run_closed = run_quiz("green-nlp-closed-book", context="closed-book")
run_full = run_quiz("green-nlp-full-context-26", context="full")
run_retrieval = run_quiz("green-nlp-retrieval", context="retrieval")
```

</details>

## Step 8 — Score: answer accuracy *and* retrieval accuracy

Same substring-match scoring as Notebook 1 for the final answers. New this session:
retrieval accuracy, the fraction of questions whose `TOP_K` retrieved documents included
the one the answer lives in. Accuracy on Notebook 1's eight questions is printed separately,
since those are directly comparable to your Session 1 and 2 rows.

### ✏️ Exercise 4 — Summarize each run

Compute each run's mean prompt length, and the retrieval run's retrieval accuracy:
the fraction of questions where the retriever fetched the document with the answer.

<details><summary><b>Hints</b> (click to open)</summary>

- Each row has the keys `prompt_tokens` and `retrieval_correct`.
- The mean of True/False values is the fraction of True.

</details>

In [ ]:
# %%solution 4   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
def score_answer(answer, reference_list):
    answer_lower = answer.lower()
    return any(ref.lower() in answer_lower for ref in reference_list)


for run in (run_closed, run_full, run_retrieval):
    for r in run["rows"]:
        r["correct"] = score_answer(r["answer"], r["reference"])
    rows = run["rows"]
    run["accuracy"] = np.mean([r["correct"] for r in rows])
    run["accuracy_session1_questions"] = np.mean([r["correct"] for r in rows[:N_SESSION1_QUESTIONS]])
    run["mean_prompt_tokens"] = np.mean([___ for r in rows])

run_retrieval["retrieval_accuracy"] = np.mean([___ for r in run_retrieval["rows"]])

for r_closed, r_full, r_ret in zip(run_closed["rows"], run_full["rows"], run_retrieval["rows"]):
    print(f"Q: {r_ret['question']}")
    print(f"   closed book  {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}")
    print(f"   full context {'✓' if r_full['correct'] else '✗'}  {r_full['answer']}")
    print(f"   retrieval    {'✓' if r_ret['correct'] else '✗'}  {r_ret['answer']}")
    if not r_ret["retrieval_correct"]:
        print(f"   retriever missed: fetched {r_ret['retrieved']}, answer is in {r_ret['source']}")
    print()

summary = pd.DataFrame({
    "closed book (no docs)": {
        "answer accuracy": run_closed["accuracy"],
        "accuracy, Notebook 1's 8 questions": run_closed["accuracy_session1_questions"],
        "retrieval accuracy": None,
        "mean prompt tokens": run_closed["mean_prompt_tokens"],
        "g CO2eq per question": run_closed["emissions_kg"] * 1000 / len(QA_PAIRS),
    },
    "full context (26 docs)": {
        "answer accuracy": run_full["accuracy"],
        "accuracy, Notebook 1's 8 questions": run_full["accuracy_session1_questions"],
        "retrieval accuracy": None,
        "mean prompt tokens": run_full["mean_prompt_tokens"],
        "g CO2eq per question": run_full["emissions_kg"] * 1000 / len(QA_PAIRS),
    },
    f"retrieval (top {TOP_K})": {
        "answer accuracy": run_retrieval["accuracy"],
        "accuracy, Notebook 1's 8 questions": run_retrieval["accuracy_session1_questions"],
        "retrieval accuracy": run_retrieval["retrieval_accuracy"],
        "mean prompt tokens": run_retrieval["mean_prompt_tokens"],
        "g CO2eq per question": run_retrieval["emissions_kg"] * 1000 / len(QA_PAIRS),
    },
})
summary

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert 0 <= run_retrieval["retrieval_accuracy"] <= 1
assert abs(run_retrieval["retrieval_accuracy"]
           - sum(r["source"] in r["retrieved"] for r in run_retrieval["rows"]) / len(QA_PAIRS)) < 1e-9
assert run_full["mean_prompt_tokens"] > run_retrieval["mean_prompt_tokens"] > run_closed["mean_prompt_tokens"]
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def score_answer(answer, reference_list):
    answer_lower = answer.lower()
    return any(ref.lower() in answer_lower for ref in reference_list)


for run in (run_closed, run_full, run_retrieval):
    for r in run["rows"]:
        r["correct"] = score_answer(r["answer"], r["reference"])
    rows = run["rows"]
    run["accuracy"] = np.mean([r["correct"] for r in rows])
    run["accuracy_session1_questions"] = np.mean([r["correct"] for r in rows[:N_SESSION1_QUESTIONS]])
    run["mean_prompt_tokens"] = np.mean([r["prompt_tokens"] for r in rows])

run_retrieval["retrieval_accuracy"] = np.mean([r["retrieval_correct"] for r in run_retrieval["rows"]])

for r_closed, r_full, r_ret in zip(run_closed["rows"], run_full["rows"], run_retrieval["rows"]):
    print(f"Q: {r_ret['question']}")
    print(f"   closed book  {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}")
    print(f"   full context {'✓' if r_full['correct'] else '✗'}  {r_full['answer']}")
    print(f"   retrieval    {'✓' if r_ret['correct'] else '✗'}  {r_ret['answer']}")
    if not r_ret["retrieval_correct"]:
        print(f"   retriever missed: fetched {r_ret['retrieved']}, answer is in {r_ret['source']}")
    print()

summary = pd.DataFrame({
    "closed book (no docs)": {
        "answer accuracy": run_closed["accuracy"],
        "accuracy, Notebook 1's 8 questions": run_closed["accuracy_session1_questions"],
        "retrieval accuracy": None,
        "mean prompt tokens": run_closed["mean_prompt_tokens"],
        "g CO2eq per question": run_closed["emissions_kg"] * 1000 / len(QA_PAIRS),
    },
    "full context (26 docs)": {
        "answer accuracy": run_full["accuracy"],
        "accuracy, Notebook 1's 8 questions": run_full["accuracy_session1_questions"],
        "retrieval accuracy": None,
        "mean prompt tokens": run_full["mean_prompt_tokens"],
        "g CO2eq per question": run_full["emissions_kg"] * 1000 / len(QA_PAIRS),
    },
    f"retrieval (top {TOP_K})": {
        "answer accuracy": run_retrieval["accuracy"],
        "accuracy, Notebook 1's 8 questions": run_retrieval["accuracy_session1_questions"],
        "retrieval accuracy": run_retrieval["retrieval_accuracy"],
        "mean prompt tokens": run_retrieval["mean_prompt_tokens"],
        "g CO2eq per question": run_retrieval["emissions_kg"] * 1000 / len(QA_PAIRS),
    },
})
summary
```

</details>

## Step 9 — Log today's runs to your leaderboard

This cell reads the `leaderboard.csv` you uploaded (one row from Session 1, or two if you
also ran Session 2) and appends today's three rows. If no file is found, it starts a fresh
one.

Today's runs answer 26 questions; Sessions 1 and 2 answered 8. Total emissions are
therefore not comparable across sessions, so the chart below divides by `n_questions`.

In [ ]:
new_rows = []
for config, run in [("closed-book-26q-fp16", run_closed),
                    ("full-context-26docs-fp16", run_full),
                    (f"retrieval-top{TOP_K}-fp16", run_retrieval)]:
    new_rows.append({
        "config": config,
        "model": MODEL_NAME,
        "params_billion": round(N_PARAMS_BILLION, 2),
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(run["elapsed"], 2),
        "energy_kwh": run["energy_kwh"],
        "emissions_kg_co2eq": run["emissions_kg"],
        "accuracy": run["accuracy"],
        "retrieval_accuracy": run.get("retrieval_accuracy"),
        "accuracy_session1_questions": run["accuracy_session1_questions"],
        "mean_prompt_tokens": round(run["mean_prompt_tokens"], 1),
    })

if os.path.exists("leaderboard.csv"):
    leaderboard = pd.read_csv("leaderboard.csv")
    print(f"Found leaderboard.csv with {len(leaderboard)} earlier row(s).")
    leaderboard = pd.concat([leaderboard, pd.DataFrame(new_rows)], ignore_index=True)
else:
    print("No leaderboard.csv found. Starting a fresh one.")
    leaderboard = pd.DataFrame(new_rows)

leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard

### Where these rows fit

Every row in your leaderboard, per question, on one chart.

In [ ]:
per_question_g = leaderboard["emissions_kg_co2eq"] * 1000 / leaderboard["n_questions"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].bar(leaderboard["config"], per_question_g)
axes[0].set_ylabel("g CO2eq per question")
axes[0].set_title("Emissions per question")

axes[1].bar(leaderboard["config"], leaderboard["accuracy"] * 100)
axes[1].set_ylabel("accuracy (%)")
axes[1].set_title("Answer accuracy")
axes[1].set_ylim(0, 105)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)
    plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

print(f"Closed book answered {run_closed['accuracy']:.0%} of the questions with no documents, "
      f"at {run_closed['mean_prompt_tokens']:.0f} prompt tokens per question.")
ratio = run_full["emissions_kg"] / run_retrieval["emissions_kg"]
print(f"On the same 26-document corpus, full context cost {ratio:.1f}x the emissions of retrieval.")
print(f"Prompt length: {run_full['mean_prompt_tokens']:.0f} vs. "
      f"{run_retrieval['mean_prompt_tokens']:.0f} tokens on average "
      f"({run_full['mean_prompt_tokens'] / run_retrieval['mean_prompt_tokens']:.1f}x).")

To keep `leaderboard.csv`: Files sidebar (folder icon, left) → right-click
`leaderboard.csv` → **Download**. Colab deletes session files when the runtime shuts down.

## Part 2 — Sparse vs. dense retrieval

Everything above is **dense** retrieval: documents and questions become vectors from a
trained embedding model, compared by cosine similarity. The older alternative is **sparse**
retrieval: represent each document by TF-IDF-weighted word counts and rank by overlap with
the question. No embedding model and no GPU, only counting words. This is the Retrieval
Playground's hand-built TF-IDF, run over the full 26-document corpus.

scikit-learn's `TfidfVectorizer` uses a smoothed variant of the playground's formula,
`idf = ln((1 + N) / (1 + df)) + 1`, and normalizes each document vector to length 1. The
ranking idea is the same.

Retrieval is fast enough here that CodeCarbon's one-second sampling would mostly measure
noise, so this part compares wall-clock time per query instead.

### ✏️ Exercise 5 — Sparse retrieval with TF-IDF

Complete `retrieve_sparse`, the TF-IDF counterpart of `retrieve`: turn the question
into a TF-IDF vector with the vectorizer fitted on the documents, and compare it to every
document's TF-IDF vector. Then mark whether the sparse retriever found the right source.

<details><summary><b>Hints</b> (click to open)</summary>

- `tfidf.transform([text])` uses the vocabulary learned by `fit_transform` above.
- `cosine_similarity(a, b)` returns a matrix; `[0]` takes its only row.

</details>

In [ ]:
# %%solution 5   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
tfidf = TfidfVectorizer(stop_words="english")
doc_tfidf = tfidf.fit_transform(DOC_TEXTS)

def retrieve_sparse(question, k=TOP_K):
    q_tfidf = tfidf.transform([___])
    sims = cosine_similarity(___, doc_tfidf)[0]
    top_idx = np.argsort(-sims)[:k]
    return [(DOC_NAMES[i], float(sims[i])) for i in top_idx]


def time_per_query_ms(retriever):
    start = time.perf_counter()
    for pair in QA_PAIRS:
        retriever(pair["question"])
    return (time.perf_counter() - start) * 1000 / len(QA_PAIRS)


comparison = []
for pair in QA_PAIRS:
    sparse_hits = [name for name, _ in retrieve_sparse(pair["question"])]
    dense_hits = [name for name, _ in retrieve(pair["question"])]
    comparison.append({
        "question": pair["question"],
        "source": pair["source"],
        "sparse": sparse_hits,
        "sparse_ok": ___ in sparse_hits,
        "dense": dense_hits,
        "dense_ok": pair["source"] in dense_hits,
    })
comparison = pd.DataFrame(comparison)

print(f"Sparse (TF-IDF) retrieval accuracy:   {comparison['sparse_ok'].mean():.0%}")
print(f"Dense (embedding) retrieval accuracy: {comparison['dense_ok'].mean():.0%}")
print(f"Sparse time per query: {time_per_query_ms(retrieve_sparse):.2f} ms")
print(f"Dense time per query:  {time_per_query_ms(retrieve):.2f} ms")

In [ ]:
# Check for Exercise 5. Run this cell as is.
assert retrieve_sparse(DOC_TEXTS[3], k=1)[0][0] == DOC_NAMES[3]
assert comparison["sparse_ok"].mean() > 0.8, comparison["sparse_ok"].mean()
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
tfidf = TfidfVectorizer(stop_words="english")
doc_tfidf = tfidf.fit_transform(DOC_TEXTS)

def retrieve_sparse(question, k=TOP_K):
    q_tfidf = tfidf.transform([question])
    sims = cosine_similarity(q_tfidf, doc_tfidf)[0]
    top_idx = np.argsort(-sims)[:k]
    return [(DOC_NAMES[i], float(sims[i])) for i in top_idx]


def time_per_query_ms(retriever):
    start = time.perf_counter()
    for pair in QA_PAIRS:
        retriever(pair["question"])
    return (time.perf_counter() - start) * 1000 / len(QA_PAIRS)


comparison = []
for pair in QA_PAIRS:
    sparse_hits = [name for name, _ in retrieve_sparse(pair["question"])]
    dense_hits = [name for name, _ in retrieve(pair["question"])]
    comparison.append({
        "question": pair["question"],
        "source": pair["source"],
        "sparse": sparse_hits,
        "sparse_ok": pair["source"] in sparse_hits,
        "dense": dense_hits,
        "dense_ok": pair["source"] in dense_hits,
    })
comparison = pd.DataFrame(comparison)

print(f"Sparse (TF-IDF) retrieval accuracy:   {comparison['sparse_ok'].mean():.0%}")
print(f"Dense (embedding) retrieval accuracy: {comparison['dense_ok'].mean():.0%}")
print(f"Sparse time per query: {time_per_query_ms(retrieve_sparse):.2f} ms")
print(f"Dense time per query:  {time_per_query_ms(retrieve):.2f} ms")
```

</details>

### A closer look at the disagreements

The questions where exactly one of the two retrievers found the right document. This is
where "matching words" and "matching meaning" come apart.

In [ ]:
disagreements = comparison[comparison["sparse_ok"] != comparison["dense_ok"]]
print(f"{len(disagreements)} of {len(comparison)} questions have exactly one retriever right.\n")

for _, d in disagreements.iterrows():
    winner = "dense" if d["dense_ok"] else "sparse"
    print(f"Q: {d['question']}")
    print(f"  correct source: {d['source']}   ({winner} got it)")
    print(f"  sparse picked:  {d['sparse']}")
    print(f"  dense picked:   {d['dense']}\n")

**Reflection:** Look at the disagreements printed above. For each one, find the answer
in the source document (`print(CORPUS["..."])`). Why might the dense retriever have missed
it? Step 4's output about the encoder's window is one clue. Does the question reuse the
document's own wording, or paraphrase it? Compare the time per query of the two methods.
What would you keep, and what would you lose, if you had to run a retriever with no GPU and
no embedding model?

## Part 3 — Retrieval-augmented translation, a simplified kNN-MT

Notebook 01e translated our Galician passage four ways. kNN-MT (Khandelwal et al., 2021)
augments a trained MT model with a large datastore of previously translated sentence pairs,
and at every generation step blends the nearest neighbors' suggested next word with the
model's own prediction.

Here is a much smaller version of the same idea, a **translation memory**: six Galician
sentences with English translations. We embed the passage we want to translate, retrieve
the most similar memory entry by cosine similarity (Part 1's mechanism, applied to
sentences), and show that entry to the LLM as a worked example before it translates.

The six entries are consecutive sentences from Chapter 1 of *Speechwriter*, the same
novel as our passage. Both translations are then evaluated with the four metrics from
Notebooks 1 and 2: BLEU, BERTScore, Qwen as a judge, and ChatGPT as a judge.

### ✏️ Exercise 6 — Retrieve from the translation memory

Complete `retrieve_translation_example`: compare the source text with every Galician
sentence in the translation memory, and return the most similar entry.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.argmax(a)` is the index of the largest value in `a`.

</details>

In [ ]:
# %%solution 6   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
TRANSLATION_MEMORY = [
    {"gl": "O meu curmán Fernando aguantaba o trebón que lle estaba a caer por riba.",
     "en": "My cousin Fernando was weathering the storm breaking over his head."},
    {"gl": "Estou rodeado de merda, carallo! Veño dar un discurso do carallo, e o carallo do discurso está na mesa dalgún pringhao papando hostias!",
     "en": "I'm surrounded by morons, damn it! I come here to give a fucking speech, and it is sitting on some loser's desk twiddling his fucking thumbs!"},
    {"gl": "Un paso máis atrás, o alcalde suaba puro medo.",
     "en": "One step behind him, the mayor was sweating pure fear."},
    {"gl": "Pero que dúas horas nin que merda! Ti toleas! Como vou ter dúas horas aí a esa xente papando hostias!",
     "en": "What do you mean, two fucking hours? Are you fucking crazy? How am I supposed to keep all those people there twiddling their fucking thumbs for two hours?"},
    {"gl": "Xuro que fun capaz de ler o pensamento do alcalde naquel momento.",
     "en": "I swear I could read the mayor's thoughts at that moment."},
    {"gl": "Á noitiña, xa máis calmo, pagou os cubatas.",
     "en": "That evening, once he had calmed down, Fernando bought me some rum and cokes."},
]

FULL_TEXT_GL = (
    "Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e "
    "refráns, dicía algo así como \"un é escravo do que di e dono do que cala\". Sempre me "
    "pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, "
    "aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida."
)
REFERENCE_EN = (
    "Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings "
    "and proverbs; she used to say something like, 'You're a slave to what you say, and "
    "master of what you keep to yourself.' It always struck me as a wise saying, and maybe "
    "I should have listened to it on a few occasions — the rare ones that, even so, ended up "
    "sharply changing the course of my life."
)

tm_embeddings = embedder.encode([pair["gl"] for pair in TRANSLATION_MEMORY],
                                normalize_embeddings=True)

def retrieve_translation_example(source_text):
    q_emb = embedder.encode([source_text], normalize_embeddings=True)[0]
    sims = tm_embeddings @ q_emb
    best_idx = int(___)  # index of the largest similarity
    return TRANSLATION_MEMORY[best_idx], sims


best_example, tm_sims = retrieve_translation_example(FULL_TEXT_GL)
for pair, sim in sorted(zip(TRANSLATION_MEMORY, tm_sims), key=lambda x: -x[1]):
    print(f"{sim:.3f}  {pair['gl']}")
print(f"\nRetrieved: {best_example['gl']}\n       -> {best_example['en']}")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert len(tm_sims) == len(TRANSLATION_MEMORY)
assert best_example == TRANSLATION_MEMORY[int(np.argmax(tm_sims))]
assert best_example["gl"].startswith("O meu curmán"), best_example["gl"]
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
TRANSLATION_MEMORY = [
    {"gl": "O meu curmán Fernando aguantaba o trebón que lle estaba a caer por riba.",
     "en": "My cousin Fernando was weathering the storm breaking over his head."},
    {"gl": "Estou rodeado de merda, carallo! Veño dar un discurso do carallo, e o carallo do discurso está na mesa dalgún pringhao papando hostias!",
     "en": "I'm surrounded by morons, damn it! I come here to give a fucking speech, and it is sitting on some loser's desk twiddling his fucking thumbs!"},
    {"gl": "Un paso máis atrás, o alcalde suaba puro medo.",
     "en": "One step behind him, the mayor was sweating pure fear."},
    {"gl": "Pero que dúas horas nin que merda! Ti toleas! Como vou ter dúas horas aí a esa xente papando hostias!",
     "en": "What do you mean, two fucking hours? Are you fucking crazy? How am I supposed to keep all those people there twiddling their fucking thumbs for two hours?"},
    {"gl": "Xuro que fun capaz de ler o pensamento do alcalde naquel momento.",
     "en": "I swear I could read the mayor's thoughts at that moment."},
    {"gl": "Á noitiña, xa máis calmo, pagou os cubatas.",
     "en": "That evening, once he had calmed down, Fernando bought me some rum and cokes."},
]

FULL_TEXT_GL = (
    "Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e "
    "refráns, dicía algo así como \"un é escravo do que di e dono do que cala\". Sempre me "
    "pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, "
    "aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida."
)
REFERENCE_EN = (
    "Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings "
    "and proverbs; she used to say something like, 'You're a slave to what you say, and "
    "master of what you keep to yourself.' It always struck me as a wise saying, and maybe "
    "I should have listened to it on a few occasions — the rare ones that, even so, ended up "
    "sharply changing the course of my life."
)

tm_embeddings = embedder.encode([pair["gl"] for pair in TRANSLATION_MEMORY],
                                normalize_embeddings=True)

def retrieve_translation_example(source_text):
    q_emb = embedder.encode([source_text], normalize_embeddings=True)[0]
    sims = tm_embeddings @ q_emb
    best_idx = int(np.argmax(sims))
    return TRANSLATION_MEMORY[best_idx], sims


best_example, tm_sims = retrieve_translation_example(FULL_TEXT_GL)
for pair, sim in sorted(zip(TRANSLATION_MEMORY, tm_sims), key=lambda x: -x[1]):
    print(f"{sim:.3f}  {pair['gl']}")
print(f"\nRetrieved: {best_example['gl']}\n       -> {best_example['en']}")
```

</details>

### Translate with and without the retrieved example

Same model and same translator prompt as Notebook 01e's Approach 4. One version adds the
retrieved memory entry to the prompt as a worked example; the other doesn't. Each
translation gets its own tracker, so the cost of the added example is visible on its own.

In [ ]:
def translate(text, example=None, max_new_tokens=200):
    system_content = (
        "You are a professional translator. Translate the user's Galician text into "
        "fluent English."
    )
    if example is not None:
        system_content += (
            " Here is one worked example of the style expected:\n"
            f"Galician: {example['gl']}\nEnglish: {example['en']}\n"
            "Output only the translation of the new text."
        )
    else:
        system_content += " Output only the translation."
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


mt_runs = {}
for label, use_example in [("plain", False), ("retrieval-augmented", True)]:
    tracker = make_tracker(f"green-mt-{label}")
    tracker.start()
    start_time = time.time()
    example = retrieve_translation_example(FULL_TEXT_GL)[0] if use_example else None
    translation = translate(FULL_TEXT_GL, example=example)
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    mt_runs[label] = {"translation": translation, "elapsed": elapsed,
                      "emissions_kg": emissions_kg, "energy_kwh": energy_kwh_of(tracker)}
    print(f"--- {label} ({elapsed:.2f} s, {emissions_kg * 1000:.4f} g CO2eq) ---")
    print(translation, "\n")

**Reflection:** Compare the two translations. Did the retrieved example change the
phrasing? Did it move toward "I had an aunt who...", the fix from Session 1's lecture? Look
at the similarity scores printed earlier: how close was the retrieved entry to the passage,
and what would a translation memory need to contain to help with *this* sentence? What
changes, besides scale, between a memory of six examples and a kNN-MT datastore of
millions?

### Evaluate the two translations: four metrics

The same four evaluation methods as Notebooks 1 and 2, from cheapest to most expensive,
each with its own tracker: BLEU, BERTScore, Qwen as a judge, and an independent judge run
by hand. Both translations are scored side by side, so each metric shows its own verdict
on the retrieved example. The reference is the same `REFERENCE_EN` used throughout the
course.

In [ ]:
CANDIDATES = {
    "general-llm": mt_runs["plain"]["translation"],
    "general-llm-retrieval-augmented": mt_runs["retrieval-augmented"]["translation"],
}
eval_runs = {}


def measure_metric(metric, score_all):
    tracker = make_tracker(f"green-eval-{metric}")
    tracker.start()
    start_time = time.time()
    try:
        scores = score_all(CANDIDATES)
    finally:
        elapsed = time.time() - start_time
        emissions_kg = tracker.stop()
    eval_runs[metric] = {"scores": scores, "elapsed": elapsed, "emissions_kg": emissions_kg,
                         "energy_kwh": energy_kwh_of(tracker)}
    for name, score in scores.items():
        print(f"{metric:16s} {name:32s} {score}")
    print(f"({elapsed:.2f} s, {emissions_kg * 1000:.6f} g CO2eq for both translations)")

#### BLEU (2002): count overlapping n-grams

Counts the word sequences a candidate shares with the reference. No model, no GPU.

In [ ]:
measure_metric("bleu", lambda cands: {
    name: round(sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score, 2)
    for name, text in cands.items()
})

#### BERTScore (2020): a learned, neural metric

Matches candidate and reference words by cosine similarity of their contextual embeddings,
so a correct paraphrase can score well even without the reference's exact words. Like
BLEU, it never looks at the Galician source. Same small `distilbert-base-uncased` model as
Notebooks 1 and 2. The cell is fail-soft: if the model download fails, the notebook
continues without a BERTScore row.

In [ ]:
def bertscore_all(cands):
    from bert_score import score as bertscore
    _, _, f1 = bertscore(list(cands.values()), [REFERENCE_EN] * len(cands),
                         model_type="distilbert-base-uncased", verbose=False)
    return {name: round(value, 4) for name, value in zip(cands, f1.tolist())}


try:
    measure_metric("bertscore", bertscore_all)
except Exception as e:
    print(f"BERTScore failed in this environment ({type(e).__name__}: {e}). Continuing without it.")

#### LLM-as-a-judge: Qwen grades the translations

Same judging prompt as Notebooks 1 and 2: the model sees source, reference and candidate,
and returns a score from 1 to 10. The judge is the same Qwen model that produced both
translations, so it is grading its own work, a known weak spot of LLM-as-a-judge.

In [ ]:
JUDGE_INSTRUCTIONS = (
    "You are an expert translation evaluator. Given a source text, a candidate "
    "translation, and a reference translation, rate the candidate's quality from 1 "
    "(very poor) to 10 (excellent), considering both accuracy and fluency. Output "
    "ONLY the number."
)


def judge_request(source, candidate, reference):
    return (
        f"Source (Galician): {source}\n\n"
        f"Reference translation: {reference}\n\n"
        f"Candidate translation: {candidate}\n\n"
        "Score (1-10):"
    )


def parse_score(text):
    match = re.search(r"\d+(\.\d+)?", str(text))
    return float(match.group()) if match else None


def qwen_judge(candidate):
    messages = [
        {"role": "system", "content": JUDGE_INSTRUCTIONS},
        {"role": "user", "content": judge_request(FULL_TEXT_GL, candidate, REFERENCE_EN)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    reply = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_score(reply)


measure_metric("llm_judge_qwen", lambda cands: {
    name: qwen_judge(text) for name, text in cands.items()
})

#### A second, independent judge: ChatGPT, by hand

To check the self-grading problem, ask a model that played no part in the translations:
ChatGPT (or Claude, or Gemini, whichever you have access to), outside this notebook. The
cell below prints one ready-to-paste prompt per translation, with the same instructions
Qwen received. Paste each into a **new** chat, so the second score is given without the
first in view.

In [ ]:
for name, text in CANDIDATES.items():
    print(f"===== prompt for: {name} =====")
    print(JUDGE_INSTRUCTIONS + "\n\n" + judge_request(FULL_TEXT_GL, text, REFERENCE_EN))
    print()

Replace each `None` below with the number your chat tool returned. Scores left as `None`
are skipped, and the rest of the notebook still runs.

In [ ]:
CHATGPT_JUDGE_SCORES = {
    "general-llm": None,
    "general-llm-retrieval-augmented": None,
}

filled = {name: float(s) for name, s in CHATGPT_JUDGE_SCORES.items() if s is not None}
if len(filled) < len(CHATGPT_JUDGE_SCORES):
    print("Some ChatGPT scores are still None; those are skipped.")
eval_runs["llm_judge_chatgpt"] = {"scores": filled, "elapsed": None,
                                  "emissions_kg": None, "energy_kwh": None}
filled

### Compare the four evaluation methods

Scores for both translations, and what each metric cost to run. The ChatGPT judge has no
cost bar: its cost was real but happened on someone else's hardware.

In [ ]:
eval_table = pd.DataFrame([
    {"metric": metric,
     **{name: run["scores"].get(name) for name in CANDIDATES},
     "duration_sec": run["elapsed"],
     "g_co2eq": None if run["emissions_kg"] is None else run["emissions_kg"] * 1000}
    for metric, run in eval_runs.items()
])
display(eval_table)

measured = eval_table.dropna(subset=["g_co2eq"])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(measured["metric"], measured["g_co2eq"])
ax.set_yscale("log")
ax.set_ylabel("g CO2eq (log scale)")
ax.set_title("What grading two translations cost")
plt.tight_layout()
plt.show()

### Log the translation and its scores

One new row in `mt_leaderboard.csv` (translation and cost) and one row per metric in
`mt_eval_leaderboard.csv`, for the retrieval-augmented translation. The plain translation
is Session 1's `general-llm` configuration again, which those files already contain. Upload
both files first (Files sidebar) if you have them; otherwise fresh files are started.

### ✏️ Exercise 7 — Append rows to a CSV log

Complete `append_rows`: when the file exists, add `rows` (a list of dicts) below the
rows already in the table.

<details><summary><b>Hints</b> (click to open)</summary>

- `pd.DataFrame(list_of_dicts)` makes one row per dict; `pd.concat` stacks tables.

</details>

In [ ]:
# %%solution 7   ← stuck? Delete the "# " at the start of this line and run the cell: the solution replaces the code.
def append_rows(path, rows):
    if os.path.exists(path):
        table = pd.read_csv(path)
        print(f"Appending to {path} ({len(table)} earlier row(s)).")
        table = pd.concat([___, pd.DataFrame(___)], ignore_index=True)
    else:
        print(f"No {path} found. Starting a fresh one.")
        table = pd.DataFrame(rows)
    table.to_csv(path, index=False)
    return table


augmented = mt_runs["retrieval-augmented"]
AUGMENTED = "general-llm-retrieval-augmented"

mt_leaderboard = append_rows("mt_leaderboard.csv", [{
    "task": "translation",
    "era": AUGMENTED,
    "model": MODEL_NAME,
    "duration_sec": augmented["elapsed"],
    "energy_kwh": augmented["energy_kwh"],
    "emissions_kg_co2eq": augmented["emissions_kg"],
    "bleu_score": eval_runs["bleu"]["scores"][AUGMENTED],
    "config": "retrieval-augmented-fp16",
    "translation": augmented["translation"],
}])

mt_eval_leaderboard = append_rows("mt_eval_leaderboard.csv", [
    {"metric": metric, "candidate": AUGMENTED, "score": run["scores"][AUGMENTED],
     "duration_sec": run["elapsed"], "energy_kwh": run["energy_kwh"],
     "emissions_kg_co2eq": run["emissions_kg"]}
    for metric, run in eval_runs.items() if AUGMENTED in run["scores"]
])

In [ ]:
# Check for Exercise 7. Run this cell as is.
assert mt_leaderboard.iloc[-1]["config"] == "retrieval-augmented-fp16"
assert len(pd.read_csv("mt_leaderboard.csv")) == len(mt_leaderboard)
assert (mt_eval_leaderboard["candidate"] == AUGMENTED).sum() >= 3
print("✓ Exercise 7 passed")

<details><summary><b>Solution to Exercise 7</b> (open it after you have tried)</summary>

```python
def append_rows(path, rows):
    if os.path.exists(path):
        table = pd.read_csv(path)
        print(f"Appending to {path} ({len(table)} earlier row(s)).")
        table = pd.concat([table, pd.DataFrame(rows)], ignore_index=True)
    else:
        print(f"No {path} found. Starting a fresh one.")
        table = pd.DataFrame(rows)
    table.to_csv(path, index=False)
    return table


augmented = mt_runs["retrieval-augmented"]
AUGMENTED = "general-llm-retrieval-augmented"

mt_leaderboard = append_rows("mt_leaderboard.csv", [{
    "task": "translation",
    "era": AUGMENTED,
    "model": MODEL_NAME,
    "duration_sec": augmented["elapsed"],
    "energy_kwh": augmented["energy_kwh"],
    "emissions_kg_co2eq": augmented["emissions_kg"],
    "bleu_score": eval_runs["bleu"]["scores"][AUGMENTED],
    "config": "retrieval-augmented-fp16",
    "translation": augmented["translation"],
}])

mt_eval_leaderboard = append_rows("mt_eval_leaderboard.csv", [
    {"metric": metric, "candidate": AUGMENTED, "score": run["scores"][AUGMENTED],
     "duration_sec": run["elapsed"], "energy_kwh": run["energy_kwh"],
     "emissions_kg_co2eq": run["emissions_kg"]}
    for metric, run in eval_runs.items() if AUGMENTED in run["scores"]
])
```

</details>

### The general-purpose LLM across all three sessions

The same passage translated by the same Qwen model three ways: as-is (Session 1), quantized
(Session 2), and with a retrieved example (today). One panel per metric, using the latest
row of each in `mt_eval_leaderboard.csv`. Panels only show the sessions whose files you
uploaded.

In [ ]:
ERAS = ["general-llm", "general-llm-quantized", AUGMENTED]
ERA_LABELS = ["Session 1\nfp16", "Session 2\nquantized", "Session 3\n+ retrieval"]

scores = mt_eval_leaderboard.copy()
scores["value"] = scores["score"].apply(parse_score)
scores = (scores[scores["candidate"].isin(ERAS)]
          .groupby(["metric", "candidate"])["value"].last()
          .unstack("candidate").reindex(columns=ERAS))
display(scores.rename(columns=dict(zip(ERAS, [l.replace("\n", " ") for l in ERA_LABELS]))))

fig, axes = plt.subplots(1, len(scores), figsize=(3.2 * len(scores), 3.5), squeeze=False)
for ax, (metric, row) in zip(axes[0], scores.iterrows()):
    ax.bar(ERA_LABELS, row.values)
    ax.set_title(metric)
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()

**Reflection:** Do the four metrics agree on whether the retrieved example helped? Where
they disagree, which one would you trust for *this* passage, and why? Compare the Qwen
judge's scores with ChatGPT's: what might explain the difference? Look at the cost of each
metric: which would you use to evaluate a thousand translations, and what would you lose?

## Wrap-up: everything measured today

Every tracked run from this notebook, in g CO2eq, on a log scale.

In [ ]:
wrapup_rows = [
    ("Build the index (embed 26 docs, once)", emissions_index_kg),
    (f"QA, full context ({len(QA_PAIRS)} questions)", run_full["emissions_kg"]),
    (f"QA, retrieval top-{TOP_K} ({len(QA_PAIRS)} questions)", run_retrieval["emissions_kg"]),
    ("Translation, plain", mt_runs["plain"]["emissions_kg"]),
    ("Translation, retrieval-augmented", augmented["emissions_kg"]),
] + [(f"Evaluation: {metric} (2 translations)", run["emissions_kg"])
     for metric, run in eval_runs.items() if run["emissions_kg"] is not None]
wrapup = pd.DataFrame([{"run": r, "g_co2eq": kg * 1000} for r, kg in wrapup_rows])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(wrapup["run"], wrapup["g_co2eq"])
ax.set_xscale("log")
ax.set_xlabel("g CO2eq (log scale)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
wrapup

**Key takeaways**

- The closed-book run shows what the model knows by itself. Every point of accuracy above
  it comes from the documents in the prompt, which is what retrieval pays for.
- On the same 26-document corpus, retrieval shrank the prompt by roughly an order of
  magnitude. Input length is the cost lever from the transformer lecture, and Step 9 shows
  how much of that reached the energy bill on your run.
- Retrieval accuracy and answer accuracy are separate numbers. A wrong answer can come from
  the retriever missing the right document or from the model misreading the right one.
  Full-context prompting has only the second failure mode.
- Sparse and dense retrieval disagree for a structural reason: TF-IDF matches words,
  embeddings match meaning. Which one wins depends on how closely a question's wording
  matches its document, and TF-IDF is nearly free to run.
- A translation memory helps as much as its closest entry is close. Six sentences rarely
  contain a near match; kNN-MT works at the scale of millions of entries.
- The four MT metrics can disagree about the same pair of translations, and they differ in
  cost by orders of magnitude. On one passage with one reference, read the translations
  before trusting any single score.
- `leaderboard.csv` now has a row for each lever this course covered: the baseline, a
  change in precision (if you ran Notebook 2), and a change in what the model reads. The
  MT leaderboards carry the same arc on the translation side.

## Share-out (discussion)

Before we wrap up, compare notes with your neighbor.

**On retrieval-augmented QA (Part 1):**

- Which questions did the closed-book model get right with no documents at all? Which kinds
  of facts did it miss, and does the publication year of the source matter?
- What was your retrieval accuracy versus your answer accuracy? Did one lag behind the
  other?
- Pick a question where retrieval missed. What did the retriever fetch instead, and why
  might it have looked similar to the question?
- How much cheaper per question was retrieval than full context? How does that compare to
  the ratio of prompt lengths?

**On sparse vs. dense retrieval (Part 2):**

- Pick a disagreement between TF-IDF and embeddings. Which would you trust more for *that*
  question, and why?
- With zero budget for a GPU or an embedding model, could you still get reasonable
  retrieval? What would you lose?

**On retrieval-augmented translation (Part 3):**

- Did the retrieved example change the translation's phrasing? Toward "I had an aunt
  who..."?
- What would a translation memory need to contain to reliably help with this passage?
- Did BLEU, BERTScore, the Qwen judge and your ChatGPT judge agree on which translation
  was better? How did ChatGPT's scores compare to your neighbor's?

**Looking back across all three sessions:**

- Which lever (precision, retrieval, or choosing a task-specific model from Session 1)
  gave the biggest measured improvement on your own numbers? Which would you reach for
  first in a system you had to build?

## Code literacy check

Before you're done, pick two or three code cells above that you ran and read today, from
any part, and add `#` comments explaining in your own words what each meaningful line or
block does and *why* it's there, so that a classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini, whichever you have
   access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, check the code again: who's
   right?

This is a self-check on your own understanding: a way to find where your mental model of
the code breaks down against a second reader.